# Partial rotary embeddings on queries and keys

RoPE rotates pairs of query and key features by angles proportional to position, so the dot product $q_m \cdot k_n$ depends only on $m - n$. Several model families rotate only the **first** `rotary_dim` features of each head and pass the rest through unchanged ("partial rotary").

Write `rope_qk(q, k, q_pos, k_pos, rotary_dim, base=10000.0)` in PyTorch.

## Inputs
- `q`: `float64` `(B, H, T_q, D)`; `k`: `float64` `(B, H_kv, T_k, D)`. $H_{kv}$ divides $H$.
- `q_pos`: `int64` `(T_q,)`, `k_pos`: `int64` `(T_k,)`: the absolute positions of the query and key tokens (shared by the batch). They can start anywhere, e.g. when queries are new tokens after a cache.
- `rotary_dim`: even int, $2 \le$ `rotary_dim` $\le D$.

## Rotation (applied to `q` with `q_pos` and to `k` with `k_pos`)
Let $r$ = `rotary_dim`. Use the **half-split** layout on the first $r$ features: for $i = 0, \dots, r/2 - 1$, features $i$ and $i + r/2$ form a pair, rotated by angle $p\,\theta_i$ with $\theta_i = \text{base}^{-2i/r}$ (note: $r$, not $D$):

$$\begin{aligned} u' &= u\cos(p\theta_i) - v\sin(p\theta_i) \\ v' &= u\sin(p\theta_i) + v\cos(p\theta_i) \end{aligned} \qquad (u, v) = (x_i,\ x_{i + r/2})$$

Features $r, \dots, D-1$ are copied unchanged.

## Output
A tuple `(q_rot, k_rot, scores)`:
- `q_rot`, `k_rot`: same shapes as `q` and `k`.
- `scores`: `(B, H, T_q, T_k)`, $\;S = q_\text{rot}\, k_\text{rot}^\top / \sqrt{D}$, where query head $h$ uses key head $\lfloor h / (H / H_{kv}) \rfloor$.

No masking, no softmax. Do not modify the inputs.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import math
import torch

def rope_qk(q, k, q_pos, k_pos, rotary_dim, base=10000.0):
    """Return (q_rot, k_rot, scores) with partial half-split RoPE on the first rotary_dim features."""
    D = q.shape[-1]
    scores = q @ k.transpose(-1, -2) / math.sqrt(D)      # TODO: no rotation, no GQA head sharing
    return q, k, scores


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[1.0, 0.0, 0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}, {"__nd__": [0, 1], "dtype": "int64"}, 4, 1.0], "e": {"__tup__": [{"__nd__": [[[[0.5403023058681398, 0.0, 0.8414709848078965, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0, 0.0, 0.0], [-0.8414709848078965, 0.0, 0.5403023058681398, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.2701511529340699, -1.1709793173055074e-17]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[1.0, 2.0, 3.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 2.0, 3.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [5], "dtype": "int64"}, {"__nd__": [5], "dtype": "int64"}, 2, 10000.0], "e": {"__tup__": [{"__nd__": [[[[2.2015107347895033, -0.39159990373668596, 3.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [[[[2.2015107347895033, -0.39159990373668596, 3.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [[[[15.0]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752, -0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [-0.22364535840745078, 0.8338841795521574, 0.5840637316429703, 0.6382860242561829, -1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736, 0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555]], [[-0.14699870178042784, 0.8925351116148221, 0.12449694399871385, 1.5847459143396285, -0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [-1.1812660464710163, 1.013579500325613, 0.5174404713814393, -0.1625051850123894, 0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304], [0.18878846665371848, 0.9860894628834065, -0.33963024501217354, 0.8497738099419226, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181]], [[0.07865408332523142, -0.19726266289721334, -0.26767571670742146, 1.3401946901039739, 1.4237928368424078, -1.2156310049465249, 1.1741144932558947, -1.723069612870992], [-0.7344792508370456, -0.1734455959217569, 0.1110606026713016, -1.0474166760042456, -0.7224687794067585, 0.07665381874343688, -0.6934227233827154, -0.6412007892602988], [0.6662634522239711, 0.9317210528359195, -0.0600550310800827, 0.018197912592243334, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706]], [[0.6483887219942397, -0.08415967031156173, -0.13743939809669994, 0.20054261321582875, 0.5306748425469374, 1.4210345023787507, -0.27354581958930024, -0.4011722662804173], [0.20057350347744685, -0.8455026583233285, 1.2724301176555226, 0.9262440047595556, 0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168], [0.4999059994351346, -0.2799827475865304, -1.6850729956882953, 0.9469623538219463, -0.3532509656510693, -0.8432343818323768, 0.8905983820560427, 0.33817143735531163]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723, -0.1382491179542964, 1.4189004130302252], [-1.0833898591198583, -2.940431935506099, -1.3840283315854316, 0.263069762783062, 0.809592098994733, 1.7335425693272173, 0.40093512304012024, -0.8023103035286667], [0.8843895901120883, -0.011057745469750928, -0.2009579446051938, 0.017552546089036017, 1.2745760509101105, -0.9014413693567734, -1.0239272781381537, -0.06652210021038106], [-0.06656995155514676, -0.8144586293391828, -0.8794565088883497, 0.37765916004244404, 1.337835134332041, 0.7129918729459087, 0.6851211109663922, 1.2457373766175497], [-1.99097272670123, -1.5103268361371722, -0.7053981939290472, -1.1951111440348927, -0.9893977243928588, 0.06987400383657381, -1.2843896578479024, -0.44349701839532474]], [[-2.0431546819494004, -0.9739745699555712, 1.345547885926575, -0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.21453528542881106, 0.6974051956994979, -0.6355527655967318, -1.4401713176798854, -1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992], [-0.39880631402193534, -0.1796813723272528, 1.0997568665514252, -0.5754988975296205, 1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], [0.5178268740164172, -0.8555092830790272, -0.10903398836430599, -0.7089297260980899, 0.8510212769423531, 0.5397406470269119, -0.7131092966031697, 0.3971833443260635], [-0.3671460370746888, -0.07191657774926997, 0.06461484502610823, -0.5640442432917747, 0.4250072828482813, 0.46028239783009683, 0.42593165582019393, 1.1046397070960905]]]], "dtype": "float64"}, {"__nd__": [4, 5, 6], "dtype": "int64"}, {"__nd__": [0, 1, 2, 3, 4], "dtype": "int64"}, 8], "e": {"__tup__": [{"__nd__": [[[[-0.27031973370803597, 1.7029522772660086, -1.7527926583040547, 1.69093127029197, -0.24059201654458218, -0.1485436170050799, -0.873748635476659, -1.076061418017875], [-1.6886326389046649, 1.4849615740122348, 0.5056760122045325, 0.6334336390553049, -0.2662940892579716, -0.9788635879610308, 1.5810523450329632, 0.9720646712384953], [2.2718403100295195, 0.8772048175119791, -0.9733850945293789, 1.285232638746736, -0.006658327042458501, 0.8604160691534417, -0.8798246019224221, 0.010491760222530112]], [[-0.4900479869219259, 0.6386232993434522, 0.10444898518407933, 1.5881905813288708, 0.6174866563552461, 0.7814837352651836, 0.5034219066562888, -0.8579926583616665], [0.04308796330218484, 1.2132768693183682, 0.5048813603955593, -0.15947753283529137, 1.244611804960027, -0.1067342450761194, 0.26391157376735364, -0.605931653133147], [-0.10770224789790256, 0.9767488949317421, -0.2571836600561672, 0.8503620269504126, -1.0457579009547568, 0.3186862865462763, -1.3826523868116858, -0.09548566256700719]], [[1.0261182319034754, 0.2916980664375113, -0.3144136615287898, 1.347076228632838, -0.9901787117600381, -1.1964880009415952, 1.16247115320443, -1.717695063867451], [-0.9011368397292183, -0.188962628754759, 0.14557849750566385, -1.0441975927351035, 0.49937290996856865, -0.01588419361176964, -0.6870054088820611, -0.6464298358259987], [0.6342503522352055, 1.0256059621106088, -0.02928177844663473, 0.024424015211586596, -0.20498186421077463, 0.15098390478798446, -0.5140741226981871, -1.0376167237538605]], [[-0.022199106934780555, -0.6308930898718533, -0.12639054604272992, 0.20214569366301638, -0.8375744283183665, 1.2760861319713963, -0.2788231221311922, -0.40036688859282243], [0.7223164182400997, -0.684651923852201, 1.2338762086484292, 0.9209814250296615, 0.0045054134415643265, -0.5103275676516011, 0.8022528897108692, 1.0548227898340798], [0.3812910922192257, 0.2450462138661367, -1.7354446212784798, 0.9449162921007314, -0.47886256483350587, -0.8540415181478035, 0.787952057801971, 0.3438470903200488]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723, -0.1382491179542964, 1.4189004130302252], [-1.2666062998704084, -3.0988075011437526, -1.3879684151537437, 0.26387194141800185, -0.474216653784994, 1.4313287106283321, 0.3870750238054049, -0.8020468326546103], [-1.527004653586572, 0.16825142683991548, -0.18044057400227784, 0.017685555095680185, 0.27376238707534917, -0.8856693928825932, -1.0277413904621535, -0.06648686209745029], [-0.12289155240106069, -0.9887855531317786, -0.8996113335628432, 0.37392025405346146, -1.3338410967771768, 0.4404581702158078, 0.6584330916981544, 1.2468647465842213], [0.5526079554491892, -1.4183133557062433, -0.6534720633588287, -1.1933275998155382, 2.153486638703887, -0.5237907534441304, -1.3115706872275579, -0.4482739022522058]], [[-2.0431546819494004, -0.9739745699555712, 1.345547885926575, -0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.9759981422360244, 0.7493103096762124, -0.640879185865463, -1.4399662886413882, -0.37172851707506455, -0.48242046733187754, 0.5294464817109109, -0.2057490558728063], [-1.2082880116493926, 0.06778534147706472, 1.1113808364928957, -0.5772077506943578, -0.9915694481345052, -1.238819897321554, -0.5701230666082194, 0.8538499440435483], [-0.6327408492755406, -0.9768035024640002, -0.08759485669866685, -0.710118084162369, -0.7694289459854466, 0.26281365463286444, -0.7160589505064755, 0.3950547710142426], [0.5616292372428473, -0.24548196294387548, 0.04753043671670973, -0.5684582779894151, 5.373774618572247e-05, 0.39594252837578847, 0.42817486055699133, 1.1023746990335166]]]], "dtype": "float64"}, {"__nd__": [[[[0.5205795406887762, -0.5760795667659703, 0.7356423186957186, -0.38998805743363774, -0.7951724350797716], [1.6914646212486064, -1.56975735314935, 0.6550851848885487, 0.24686788406683927, -2.3673177919004837], [-0.6912133244324353, -1.0677060730724541, -1.0548208582382335, 0.011073276425358674, -0.07145509091413423]], [[-0.20440418878029112, 0.22082387169500325, -0.04187410700647909, -0.45576388633725556, -0.8822220968039366], [-1.3043682670883257, -1.6659351866891834, 0.08793979870302757, -1.4169245070638352, 0.2916932166265366], [1.0196574653926402, -0.6418891096722795, 0.44161098202141047, 0.036265552579717766, -1.0091303968615446]], [[-2.2162878692735473, 0.4935559494719552, -0.711487666491901, -1.0347813050390482, -0.7585862108746003], [1.0258393986052388, -0.006889689093097095, 0.4259520477823143, 0.4708170587852169, -0.308384978261473], [-0.9388525151280203, 0.4652035765823827, -0.46674300039679223, -0.4462971812178517, -0.42957307528803373]], [[0.3242881917619469, -0.3797126211806463, -0.42649425168197086, 0.5370979897854498, -0.012035383008094692], [0.0736615452995581, -0.5206944116230691, 0.3505648532561966, -0.29899682027746405, 0.4996081370863291], [-1.7018369239780182, 0.43973697640464693, -0.5448464671731498, -0.4539594838661096, -0.030896898273836043]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752, -0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [-0.22364535840745078, 0.8338841795521574, 0.5840637316429703, 0.6382860242561829, -1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736, 0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555]], [[-0.14699870178042784, 0.8925351116148221, 0.12449694399871385, 1.5847459143396285, -0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [-1.1812660464710163, 1.013579500325613, 0.5174404713814393, -0.1625051850123894, 0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304], [0.18878846665371848, 0.9860894628834065, -0.33963024501217354, 0.8497738099419226, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181]], [[0.07865408332523142, -0.19726266289721334, -0.26767571670742146, 1.3401946901039739, 1.4237928368424078, -1.2156310049465249, 1.1741144932558947, -1.723069612870992], [-0.7344792508370456, -0.1734455959217569, 0.1110606026713016, -1.0474166760042456, -0.7224687794067585, 0.07665381874343688, -0.6934227233827154, -0.6412007892602988], [0.6662634522239711, 0.9317210528359195, -0.0600550310800827, 0.018197912592243334, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706]], [[0.6483887219942397, -0.08415967031156173, -0.13743939809669994, 0.20054261321582875, 0.5306748425469374, 1.4210345023787507, -0.27354581958930024, -0.4011722662804173], [0.20057350347744685, -0.8455026583233285, 1.2724301176555226, 0.9262440047595556, 0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168], [0.4999059994351346, -0.2799827475865304, -1.6850729956882953, 0.9469623538219463, -0.3532509656510693, -0.8432343818323768, 0.8905983820560427, 0.33817143735531163]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723, -0.1382491179542964, 1.4189004130302252], [-1.0833898591198583, -2.940431935506099, -1.3840283315854316, 0.263069762783062, 0.809592098994733, 1.7335425693272173, 0.40093512304012024, -0.8023103035286667], [0.8843895901120883, -0.011057745469750928, -0.2009579446051938, 0.017552546089036017, 1.2745760509101105, -0.9014413693567734, -1.0239272781381537, -0.06652210021038106], [-0.06656995155514676, -0.8144586293391828, -0.8794565088883497, 0.37765916004244404, 1.337835134332041, 0.7129918729459087, 0.6851211109663922, 1.2457373766175497], [-1.99097272670123, -1.5103268361371722, -0.7053981939290472, -1.1951111440348927, -0.9893977243928588, 0.06987400383657381, -1.2843896578479024, -0.44349701839532474]], [[-2.0431546819494004, -0.9739745699555712, 1.345547885926575, -0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.21453528542881106, 0.6974051956994979, -0.6355527655967318, -1.4401713176798854, -1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992], [-0.39880631402193534, -0.1796813723272528, 1.0997568665514252, -0.5754988975296205, 1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], [0.5178268740164172, -0.8555092830790272, -0.10903398836430599, -0.7089297260980899, 0.8510212769423531, 0.5397406470269119, -0.7131092966031697, 0.3971833443260635], [-0.3671460370746888, -0.07191657774926997, 0.06461484502610823, -0.5640442432917747, 0.4250072828482813, 0.46028239783009683, 0.42593165582019393, 1.1046397070960905]]]], "dtype": "float64"}, {"__nd__": [104, 105, 106], "dtype": "int64"}, {"__nd__": [100, 101, 102, 103, 104], "dtype": "int64"}, 8], "e": {"__tup__": [{"__nd__": [[[[-0.35492933863170767, -1.5097096347684449, -0.2118037902213269, 1.7899105450219286, -0.07058641112445502, -0.8018032698148372, -1.9470125668443432, -0.9018741469945559], [-1.5909819701192986, -1.7785114350579356, -1.0571917583914074, 0.5332245719558312, 0.6254351300520146, 0.013486122385561572, 1.2797579197030173, 1.0304462412078423], [1.9556792261207845, -0.2679530817750851, 0.21442466316597553, 1.277764400634616, -1.1561234761536663, -1.1991685660869031, -1.2944465752691667, 0.13874851063462104]], [[-0.10990360082731937, -0.11070697848301975, -0.3671809000273911, 1.6659125822196954, 0.7806138602332161, -1.0031453094459946, 0.3598908074159609, -0.6951517768274509], [0.6673842184752712, -1.076091760508303, 0.05071463133134704, -0.0981885822526352, 1.0514339839274447, -0.5704905640421152, 0.5674350473949926, -0.6188257057226351], [-0.622409550912719, -0.6461901211619978, 1.0245049410174631, 0.8556464187447759, -0.8472400560314577, -0.798772608628812, -0.963462860512403, -0.010114085441322362]], [[0.38344863837523047, -0.8956702740724156, -1.148064172418153, 1.5118298254199671, -1.3734408064404031, 0.8452491103240759, 0.3635158711778617, -1.574630520837478], [-0.5242020197031068, 0.14991182519330487, 0.656751515867469, -0.9744456550117581, 0.886923418152223, 0.11612763383371352, -0.2486905248999169, -0.7474461925439977], [0.4431302754005664, -0.7784183312488997, 0.41675744587646213, 0.12789081957057846, -0.49792231621933253, -0.6846375906985629, -0.30239520082738774, -1.029994629210423]], [[-0.44326162123036295, 1.223582224708142, 0.16633246370012644, 0.2411058015881986, -0.7110153714694011, -0.7275083823819821, -0.2570027530658208, -0.37818582653043953], [0.6251484657774841, 0.296842966360507, -0.00840636848140175, 0.8110737909821297, -0.36187111316813325, 0.8006664327059141, 1.4717301146225883, 1.1414977918440727], [0.08631495501044034, -0.6702279168114674, -1.6007035246432562, 0.9058682166485743, -0.6060049352490451, 0.5832916090403617, -1.0345939808028186, 0.43646350897284636]]]], "dtype": "float64"}, {"__nd__": [[[[-2.1831140288035806, -0.12665039065076866, -0.029135146969260173, 0.8459992048336056, -0.9157492894157937, 0.3773936462426623, -0.30124894341421765, 1.5109076490239541], [-1.3323455360554897, 3.3787941835021433, -1.0756349366557894, 0.3426247564255876, 0.2324399409556238, 0.4848295293665994, -0.9587976212756507, -0.7716967017703941], [-1.1781410642156709, -0.6229978289845656, 0.7673221017544238, 0.02423481159077076, 1.0092931632885493, 0.6516076436040377, -0.7071265506078286, -0.06438909533273127], [-0.7813830069882846, 1.0692803490560903, -1.0401144200104704, 0.24757344251702737, -1.087968290665326, 0.1683443046689167, -0.4012439170930696, 1.277965352905237], [1.5669759113887405, 0.9051131285197357, 0.7505762151733564, -1.1426132171022454, 1.5771704881303492, 1.2110903157690145, -1.258522447317112, -0.5651683713949234]], [[-1.597718055970329, 1.202485173962736, 1.1400827137276868, -0.597994892643674, 1.3140943928049058, -0.06432931177228263, 0.8670038301863185, 0.5219926462486209], [0.6533910685003481, -0.8911718658464265, -0.7917823542743618, -1.41223182383848, -0.8147604406566208, -0.0028553478889768447, -0.2532200847709151, -0.34847792204678607], [-1.5440262549137105, -0.7308209268548582, 1.08022364697529, -0.6595668733892167, -0.2432135146461872, 1.0025818487235614, 0.6271559194632649, 0.7919596289788915], [-0.9352367571199622, 0.9625841847699688, 0.5552151272067137, -0.7460101191337911, -0.3430948752079291, 0.3108822714761252, -0.46059683242650257, 0.3221876281074989], [0.48433070145134327, 0.42137802014334896, -0.3346159170259173, -0.6756721870157767, -0.28434340970976896, -0.19867673252663914, 0.2713393478660708, 1.0401162851226031]]]], "dtype": "float64"}, {"__nd__": [[[[0.5205795406887758, -0.5760795667659707, 0.7356423186957188, -0.3899880574336378, -0.7951724350797713], [1.6914646212486062, -1.569757353149349, 0.6550851848885489, 0.24686788406683943, -2.3673177919004833], [-0.6912133244324355, -1.0677060730724544, -1.0548208582382337, 0.01107327642535884, -0.07145509091413416]], [[-0.2044041887802912, 0.22082387169500192, -0.041874107006478894, -0.45576388633725556, -0.8822220968039365], [-1.3043682670883254, -1.665935186689184, 0.08793979870302808, -1.4169245070638352, 0.2916932166265366], [1.0196574653926402, -0.6418891096722797, 0.4416109820214103, 0.036265552579717864, -1.0091303968615446]], [[-2.2162878692735477, 0.49355594947195486, -0.711487666491901, -1.0347813050390484, -0.7585862108746], [1.0258393986052388, -0.006889689093096994, 0.42595204778231416, 0.470817058785217, -0.308384978261473], [-0.9388525151280196, 0.46520357658238287, -0.4667430003967923, -0.44629718121785167, -0.4295730752880335]], [[0.3242881917619471, -0.37971262118064614, -0.4264942516819711, 0.5370979897854495, -0.012035383008094626], [0.07366154529955818, -0.5206944116230694, 0.35056485325619624, -0.298996820277464, 0.4996081370863291], [-1.7018369239780182, 0.4397369764046469, -0.5448464671731498, -0.45395948386610974, -0.030896898273836113]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752, -0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [-0.22364535840745078, 0.8338841795521574, 0.5840637316429703, 0.6382860242561829, -1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736, 0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555]], [[-0.14699870178042784, 0.8925351116148221, 0.12449694399871385, 1.5847459143396285, -0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [-1.1812660464710163, 1.013579500325613, 0.5174404713814393, -0.1625051850123894, 0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304], [0.18878846665371848, 0.9860894628834065, -0.33963024501217354, 0.8497738099419226, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181]], [[0.07865408332523142, -0.19726266289721334, -0.26767571670742146, 1.3401946901039739, 1.4237928368424078, -1.2156310049465249, 1.1741144932558947, -1.723069612870992], [-0.7344792508370456, -0.1734455959217569, 0.1110606026713016, -1.0474166760042456, -0.7224687794067585, 0.07665381874343688, -0.6934227233827154, -0.6412007892602988], [0.6662634522239711, 0.9317210528359195, -0.0600550310800827, 0.018197912592243334, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706]], [[0.6483887219942397, -0.08415967031156173, -0.13743939809669994, 0.20054261321582875, 0.5306748425469374, 1.4210345023787507, -0.27354581958930024, -0.4011722662804173], [0.20057350347744685, -0.8455026583233285, 1.2724301176555226, 0.9262440047595556, 0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168], [0.4999059994351346, -0.2799827475865304, -1.6850729956882953, 0.9469623538219463, -0.3532509656510693, -0.8432343818323768, 0.8905983820560427, 0.33817143735531163]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723, -0.1382491179542964, 1.4189004130302252], [-1.0833898591198583, -2.940431935506099, -1.3840283315854316, 0.263069762783062, 0.809592098994733, 1.7335425693272173, 0.40093512304012024, -0.8023103035286667], [0.8843895901120883, -0.011057745469750928, -0.2009579446051938, 0.017552546089036017, 1.2745760509101105, -0.9014413693567734, -1.0239272781381537, -0.06652210021038106], [-0.06656995155514676, -0.8144586293391828, -0.8794565088883497, 0.37765916004244404, 1.337835134332041, 0.7129918729459087, 0.6851211109663922, 1.2457373766175497], [-1.99097272670123, -1.5103268361371722, -0.7053981939290472, -1.1951111440348927, -0.9893977243928588, 0.06987400383657381, -1.2843896578479024, -0.44349701839532474]], [[-2.0431546819494004, -0.9739745699555712, 1.345547885926575, -0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.21453528542881106, 0.6974051956994979, -0.6355527655967318, -1.4401713176798854, -1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992], [-0.39880631402193534, -0.1796813723272528, 1.0997568665514252, -0.5754988975296205, 1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], [0.5178268740164172, -0.8555092830790272, -0.10903398836430599, -0.7089297260980899, 0.8510212769423531, 0.5397406470269119, -0.7131092966031697, 0.3971833443260635], [-0.3671460370746888, -0.07191657774926997, 0.06461484502610823, -0.5640442432917747, 0.4250072828482813, 0.46028239783009683, 0.42593165582019393, 1.1046397070960905]]]], "dtype": "float64"}, {"__nd__": [4, 5, 6], "dtype": "int64"}, {"__nd__": [0, 1, 2, 3, 4], "dtype": "int64"}, 4], "e": {"__tup__": [{"__nd__": [[[[-1.5864098873208579, 1.442022376185535, 0.8961034075903656, 1.7456753774781795, -0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [0.49663315908821726, 0.800941036217552, 0.3801357576602946, 0.6791651714943084, -1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [1.8100263983781457, 1.130568733809742, -1.5936139497178328, 1.3555050629202519, 0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555]], [[0.19030436157044717, 0.8284482447547675, 0.02987235105268013, 1.6191701714903757, -0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [0.16110572034532566, 1.0204346640303028, 1.2795229817555938, -0.11164423443059271, 0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304], [0.08637112202968421, 0.9333591920292121, -0.37885329317273275, 0.9073745508484754, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181]], [[-0.25398939015792715, -0.2506983671448445, 0.11543891815920855, 1.331234274749225, 1.4237928368424078, -1.2156310049465249, 1.1741144932558947, -1.723069612870992], [-0.10184528160960057, -0.12087981874426737, 0.7358136761366436, -1.0547763446990974, -0.7224687794067585, 0.07665381874343688, -0.6934227233827154, -0.6412007892602988], [0.6229460634779594, 0.9289532382612182, -0.24382739084185678, 0.07403489342455896, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706]], [[-0.5278296314046469, -0.0921119171304139, -0.4008658169135168, 0.19701671133420132, 0.5306748425469374, 1.4210345023787507, -0.27354581958930024, -0.4011722662804173], [1.2770592459747827, -0.8907389060648894, 0.16860550668464047, 0.8828289204637324, 0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168], [0.009159376184069423, -0.33626258653378593, -1.7576385051816215, 0.9284694455929673, -0.3532509656510693, -0.8432343818323768, 0.8905983820560427, 0.33817143735531163]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723, -0.1382491179542964, 1.4189004130302252], [0.5792616441446045, -2.9429155689175888, -1.6594348306269184, 0.23365278011901275, 0.809592098994733, 1.7335425693272173, 0.40093512304012024, -0.8023103035286667], [-0.18530538827068885, -0.011406561513228296, 0.8878011915268085, 0.01732789553080436, 1.2745760509101105, -0.9014413693567734, -1.0239272781381537, -0.06652210021038106], [0.19001266216127705, -0.8254202258546618, 0.8612609927858819, 0.3530591321847715, 1.337835134332041, 0.7129918729459087, 0.6851211109663922, 1.2457373766175497], [0.7675395087704817, -1.4613270368322178, 1.9678521572879555, -1.2545521471727192, -0.9893977243928588, 0.06987400383657381, -1.2843896578479024, -0.44349701839532474]], [[-2.0431546819494004, -0.9739745699555712, 1.345547885926575, -0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.6507131209713302, 0.7117717988797435, -0.1628654068469627, -1.433125373990694, -1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992], [-0.8340441029138309, -0.16813622661657165, -0.8202938961283541, -0.578977189462896, 1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], [-0.4972578424975469, -0.8338596310314833, 0.18101856298989585, -0.7342721605208401, 0.8510212769423531, 0.5397406470269119, -0.7131092966031697, 0.3971833443260635], [0.28888334100891927, -0.04930329841672111, 0.23562195573613506, -0.5664689641198265, 0.4250072828482813, 0.46028239783009683, 0.42593165582019393, 1.1046397070960905]]]], "dtype": "float64"}, {"__nd__": [[[[0.909432740902158, -2.517343655229041, 0.9398712219195406, -0.9320782203620409, -0.7951724350797711], [1.6848035263526666, -2.401104333081299, -0.760626629497666, -0.39438956414017734, -0.6192716451232129], [-0.7290197380399968, 0.43583627303029093, -0.10360455846700702, -0.3703836630837471, -1.6455540471383947]], [[0.43764066506791466, -0.32383188487429543, -0.6559227440211155, -0.5251966071092798, -0.8822220968039363], [-0.7649171790925363, -1.8842631588696024, 0.7071673872365963, -0.10378095403545887, 0.28825439042347145], [1.0270020672850393, -1.2939826109993238, -0.00045777896853811694, -1.2054523902944032, -0.1346961911138619]], [[-0.6285489927425972, -0.7318600366573198, 0.30549977303122505, -0.5612095078665642, -0.7585862108746002], [0.5931638608481603, 0.599218350165722, -0.4282143062239355, 0.2564951363476515, -0.1866791391616596], [-1.139829555616977, 0.4279276993347079, -0.20319122692233124, -0.5276290429353586, -0.5469138672246135]], [[0.566429356060683, -0.7147181582729639, -0.16014053890283578, 0.48662472828794917, -0.012035383008094657], [-0.5687868700930421, -0.5504313675606938, 0.0320701349482455, -0.03333580718422169, 0.5895309014465755], [-1.2421251186015312, -0.014394837549854838, 0.429948857475245, -0.7002489483593528, -0.24969037750216883]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.27174114252366427, -1.4828593634501943, -0.489072005359709, 0.2007368892738041, 0.02198571377633105, 0.5786880417250706], [0.8064177974523544, -0.11758989750756722, -0.11037013953690167, -0.9874075351661735, 1.0330955204913965, -0.26393657528783593], [-1.0833076865189253, -1.114373881782793, -0.4147657766285512, 1.5553992101837082, 0.7725462701194257, -1.6940422362384258], [-0.12361444090580717, 0.2639596332433514, 0.6265573247505951, 0.3436628055732324, -0.6407415119480926, 0.4118864085570845]], [[-0.5541505601270842, -2.2406484235127633, -0.7088634775653162, -0.6859673154744348, 0.9536290088133706, 1.0560046751839265], [-1.045279424641642, 0.17390955321529164, 0.09013529248460428, -1.435795498095001, -2.0531366566250346, -0.09276540124128595], [0.13644372719186973, -0.5948320751120193, -2.7111664954113537, -0.029483978631708624, -0.5786198747278738, -0.8894063342725544], [-0.32418700968801495, -1.12855669323536, 0.5598427280686814, 0.010603864924009842, -0.5594566369571077, 0.2818726076342451]]], [[[-0.032100222264474815, -1.518206205158868, -0.1441467553139046, -0.4571565407824213, -1.7407156772498007, -0.6520424779536729], [0.42634016461290364, -0.6194956228417741, 1.4075074257264955, 1.6982164351447544, -0.3063453638612675, 0.553434424296176], [1.9926526478415145, -0.331526522350224, -0.8989570837450346, -0.422331995147081, -1.3742214526554224, -1.3491851888620348], [0.2086518433004105, -0.4167563978840294, 0.24644175655585768, 0.4883741105141886, -0.7643160931773055, -0.1944523423331713]], [[-0.2693043085863964, -0.6686649004393013, -0.8364556720740618, 0.3155505374569986, -0.14024643621485902, 0.3322867996571848], [1.0009222444961468, 1.0142174316108183, -1.1120591826841941, -0.8558236870860543, -0.2562514323596529, 0.1236371145358894], [0.9383854586360836, -0.9621771389420167, -0.43685307558735226, 0.24703569469092412, 1.2288651191177333, 1.3792342883219184], [-1.0059765041187922, -0.21203724299336404, -0.8077502873624303, 0.8474791280591054, 0.4469000649128899, -0.35779615802814096]]]], "dtype": "float64"}, {"__nd__": [[[[-0.6112468227277504, 0.2949523400999831, 0.9802940600621635, -0.767444144080219, -0.3236618111270047, 0.1129737681875106], [0.018011866501908936, 0.3378669966589955, -0.9995766587698955, -0.5949276168950793, -2.1001051526512886, 0.05573366268881772], [1.0701569228810817, -2.057195404042072, -0.6318079969806747, 0.8985241778631176, -0.1428027365344208, -1.8950428195675442], [1.168788331744318, -1.4259696378610343, 1.3709104657520208, -0.58464059879444, -0.7065454379067378, 0.026944605818733074]]], [[[-0.022175546391032475, 0.6004755228520038, 0.5025060567088484, -0.14793754391747488, 1.3225328287068747, 0.7134748212984285], [0.27379161855350254, 0.13511514244719608, -0.09532521278129974, -0.07758069467940808, 0.6648989933924994, 0.4742533005081589], [0.0397348380155517, 1.0151129233973155, -0.26221152794886715, 0.43589613755299295, 0.8595237833519895, -1.1155043350355576], [-0.2641148710183764, 1.6688064665782514, -0.9571035511645589, 1.1520796067501626, -1.0597628522710998, 0.405535390337305]]]], "dtype": "float64"}, {"__nd__": [0, 1, 2, 3], "dtype": "int64"}, {"__nd__": [0, 1, 2, 3], "dtype": "int64"}, 6, 500.0], "e": {"__tup__": [{"__nd__": [[[[-0.27174114252366427, -1.4828593634501943, -0.489072005359709, 0.2007368892738041, 0.02198571377633105, 0.5786880417250706], [1.2665841864796312, -0.2464756071639758, -0.10616667804554578, 0.14507861010698764, 1.0101304438182126, -0.26565526523831606], [-0.9635054327546894, -1.2717973924129022, -0.3607833112111234, -1.632323352698015, 0.4703065814198537, -1.706354344787011], [0.07387967107583712, 0.48178710474328357, 0.6062395330926271, -0.35766806977510135, -0.498101956248494, 0.44124610283639853]], [[-0.5541505601270842, -2.2406484235127633, -0.7088634775653162, -0.6859673154744348, 0.9536290088133706, 1.0560046751839265], [0.643413368354343, 0.4305262304470361, 0.09159643350018143, -1.6553359252284572, -2.015009138533628, -0.09132296528688047], [-0.029970919535201868, -0.431782071191533, -2.6815681201264927, 0.13633759447848154, -0.7086539352954453, -0.9750178513270882], [0.3194462895829287, -0.8424335743467857, 0.5457897530747343, -0.05624702012981131, -0.9364494707644326, 0.3082038165638661]]], [[[-0.032100222264474815, -1.518206205158868, -0.1441467553139046, -0.4571565407824213, -1.7407156772498007, -0.6520424779536729], [-1.1986472820736576, -0.5760901205370623, 1.398545239705309, 1.2763031339518638, -0.3817623366690368, 0.5757065467483282], [-0.4452106992831362, 0.021572346752562366, -0.8556773113458499, 1.9876660489927713, -1.413481188348681, -1.3770406134562623], [-0.2754831176812511, -0.10527545752570855, 0.25541907776115896, -0.45404175513470335, -0.8641655301560249, -0.18250026720208518]], [[-0.2693043085863964, -0.6686649004393013, -0.8364556720740618, 0.3155505374569986, -0.14024643621485902, 0.3322867996571848], [1.2609513974902091, 1.038378533499775, -1.1138816093820714, 0.37984351524313437, -0.1267746675660197, 0.10596943976168419], [-0.6151350615897976, -1.2381792333911381, -0.4804135375831258, 0.7504683600485742, 0.9501613092974366, 1.3646723653096609], [0.8763129294516812, -0.3619944190526394, -0.7898019886897175, -0.9809613901732833, 0.3371045846767946, -0.3958426910978077]]]], "dtype": "float64"}, {"__nd__": [[[[-0.6112468227277504, 0.2949523400999831, 0.9802940600621635, -0.767444144080219, -0.3236618111270047, 0.1129737681875106], [0.5103461806820879, 0.5990860792732136, -1.0003354022396007, -0.30628430018945896, -2.0410026039289884, 0.039860016814289745], [-1.2623681409375802, -1.9566239430137227, -0.5713358595262606, 0.5991729420965988, -0.6512052653341914, -1.9141431421559436], [-1.07458719252703, -1.0645714304793266, 1.3680735725360256, 0.7437292248105238, -1.1829131554495256, 0.09217492619831723]]], [[[-0.022175546391032475, 0.6004755228520038, 0.5025060567088484, -0.14793754391747488, 1.3225328287068747, 0.7134748212984285], [0.21321214638578984, 0.05049358409171209, -0.10284118852565652, 0.18847067467022743, 0.6766071024593654, 0.472680419710096], [-0.41289476338107484, 0.7687536407731675, -0.22667028667707653, -0.14526601274291961, 1.0854737312316505, -1.1232655020119389], [0.09889025715859548, 1.9421063173243285, -0.9753235891831964, -1.177822058895897, -0.36910076964983396, 0.35951363931101027]]]], "dtype": "float64"}, {"__nd__": [[[[-0.34558218953262454, -0.25356111763894384, 1.0296540456968117, 0.5626314672994219], [-0.5794109323752059, -0.6171753161923831, -0.45656508888198155, -0.9615848491539157], [0.3134814999295128, -0.5799944135306255, 2.405706172272597, -0.013027144155993542], [0.4804243897699918, 0.3525864405834813, -0.8642014253729062, 0.24534429568622163]], [[-0.27759546680566255, -1.0656156383472688, 0.9941997527146801, 0.19193340838838063], [0.7086104593177345, 2.0864185513180726, -0.49470609416840217, 0.048835638723618374], [-1.1117333819671162, 1.5408268588642255, 1.9694869998133206, -0.9499574937202334], [0.19284607955011884, 0.42995630189006667, 0.3753477800649864, 0.9775721159826473]]], [[[-1.5036222774718215, -0.6698654344569606, -0.9029936810869763, -0.761208388777391], [0.041019092862892466, -0.07108225435220222, -0.6170397753769865, -1.5336956732416156], [-1.4505331564134971, -0.5056118241449827, 0.04821853329887966, -0.6050377445572078], [-0.4632326071259465, -0.34572836249936895, -0.28257197849536453, 0.1254621472382889]], [[-0.3310701756486049, 0.047555051226376, -0.32029669969539687, -0.2898034715122916], [-0.0458968641195889, 0.1925857906063155, 0.08911250732401288, 1.1697270218864313], [0.46866510654786186, 0.5246451853189674, -0.48969589332777924, -1.1189893972353055], [-0.13274302721864845, 0.04322702405626801, 0.20084593441848708, 0.4256401695228964]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-1.0641892166186686, -0.6163888649025725, -0.37531224638067917, -0.5488203633007783, 1.0085690589142688, 0.8334942966216093, 0.8700599664307159, 0.19348830714441678, 1.6366916490384464, 1.0287923702896826, 0.3247528649959841, -1.0487857298833259, 0.6854466141613622, 0.6521584987897013, 0.09434243617109991, 0.7128873475063768], [0.7699365242725695, -0.29845955358926546, 0.5652951110659917, 0.347065490112703, 0.3131496577718769, -0.34179643371544444, -0.46297793464524145, 0.11862428149034983, 1.4317178577246314, 2.325985422686139, -0.3698057478971567, -1.0543470939889712, -0.34156052722203584, -0.8351430939597624, -0.4170599012110719, 0.3967821823193485]], [[-0.9142515319015107, 0.6089667102418319, 0.7915542794734144, 1.5164360247500759, 1.8192748316879657, 0.997828907708521, 2.2671920580788893, -0.49962791416033, 0.276470890505262, 0.17205662638670258, 0.43919349351729436, 1.024591077762065, -0.3232677308941149, 1.7133716327948385, 1.2759740433101456, 0.35307602330793086], [0.47171827985954007, -0.9634178520124883, -2.4759797049869987, -0.0809123582275483, 0.3239967842305745, 0.23537058308694503, -1.0897890047921537, -0.4421162864665036, 1.759257542559709, 0.7362004871113208, 0.6243032398185351, -0.2989905966535996, 0.7467517137104908, 1.2923755482918649, 0.7663819416213754, -0.3931861881713645]], [[0.29516001191068375, -0.8641129705485112, -0.2917449236407673, 0.6010780516446905, -0.5195588624307894, -0.21733011358290186, 1.4400312683510301, -0.15393592947116394, 0.7754822078311886, -0.2849522608169767, -1.1168312675166585, 0.0183330348372163, -0.34331274239099885, -1.983856891778598, -1.1801191577307932, -0.4033017548097564], [1.4802541598391297, -0.5561659964139166, -0.017331883732870673, 0.6397451788405071, 0.3037293059107359, -0.32103423554708943, 1.7031759481970714, 1.3443712426623422, -2.4063036979730765, 0.9680702588577331, 0.5544719540727608, 0.9869177298341354, 0.10338523921171394, 0.8337698290032067, 1.9270332857269237, 1.7130171495621231]], [[1.7902198650737964, 1.0600020830044186, 0.9309145546739234, 0.9497811075000052, -1.09442440901382, 2.269770121481706, 0.2510860761421567, 0.3376584232414907, 0.13434967640702897, 0.4856635218825325, -2.088412931573151, 0.24372955871343904, -1.2410871583938738, -0.9626480973950894, -1.8295130964672903, -1.6892691709142154], [0.5960192712221675, -0.47166994919225386, -0.08914065036402882, -0.15682525475683184, -1.1393991436194493, -0.6911048845295463, -1.2390976458361571, -0.2978979313811222, -2.194688737213589, -0.3814903772748522, 0.912731508619598, 1.20625446188279, 1.1438534923955528, 0.498781081889901, -0.9924858329137045, 0.9059666097190682]], [[0.520598941722906, 0.2684169778476837, -1.2004373031856796, 0.3919140032283127, 1.4058874357213444, 0.7403878069322796, -0.22429223776016513, 0.5448080755421495, 0.006254860766122802, 0.17954288578318953, 0.8423605426473773, 0.3437347798979491, 0.07454426859255363, 0.38411597350359655, -0.8126967496866394, -0.988282585949805], [0.483228178679919, -0.5877535495917325, 0.5570833818970693, -1.827723926934063, 1.1912715587266225, -0.09833773810578231, 1.6563475651929895, -2.4594146469559424, -1.670335623109287, 1.3966502013370818, 0.15800957363487542, 0.3047770205792349, 0.12683901406770667, 0.16268457466540617, 0.5384943889153253, -0.9986363697499262]], [[-0.16156207823471475, 1.2970132224835127, 0.1847480249612491, -0.6114204248758205, 0.1007369315712793, 1.0435456977369353, -0.1924031722288013, -0.17070867614992088, 0.49404902758840713, 0.8514257825901366, 1.4736863204876074, -0.12171205912006818, 0.3532571718128816, 1.045571670839754, -1.8825315671085434, 0.15230404541692436], [0.46941052441393144, -0.07183858457688504, -0.8355094739653411, 0.5943599030468357, 1.2489836306385804, -0.07383949408338283, -0.46514749878761874, -0.19825180317926416, 1.5646532222165077, 1.2544015941713493, 0.061488566602324234, -1.071458094300794, -1.238651024094895, -0.29998801234913414, -0.2925278571670005, 0.2874945668536115]]]], "dtype": "float64"}, {"__nd__": [[[[-1.1024516074091504, -0.10114598111265312, 0.5541566905155195, -1.6555993696768185, 2.3107534963187315, 0.5990995406906326, 0.5504347634351624, -1.270088079239614, -0.993937507139367, 1.0423765305690167, 0.9134012445241502, -2.396483760058587, 1.1814470254959588, 0.7569620938110935, 0.653281059632514, -1.074377365542198], [0.2906301224898339, -0.5702189495545404, 0.5951640454677188, 1.8715370200473402, 0.6760785806750421, 1.0515261192899155, -0.272888752787871, -0.9806365313800909, 0.0539999690628915, 1.0759170532855784, 1.5076469684093838, -0.01707683380754581, 0.29611499026283156, -0.35429405231554517, -1.6708150408584765, 0.13019258819129437], [0.39184768673802084, -0.36842813509906974, -0.12693278704905994, -1.7515381952417695, 0.10273510738944559, 1.4258622114365382, 0.02320020698432567, 0.6410914325288839, 0.4896998661726692, -0.709506693902454, 0.25619616215093155, -0.7707925907281221, 2.9585937751357627, 0.252086333704382, -1.040342565011859, -1.4854231088889078], [1.5358514009320587, 0.659798205263123, 0.4425572015525159, -1.3125886957580712, -0.5153511413566457, -1.1656481403328238, 1.1424286882662678, -0.47732331799931776, 1.9688767327737233, 0.5635815939087825, -0.27574887506311774, 1.3840476434006113, -2.049071153160133, -0.7857514616136716, -0.35565023747576796, 1.1121408717949874], [1.1199356655285706, 0.4008160818715742, -1.727570308196217, -0.48281287877856743, -1.0563636699135825, 1.291607523146258, -0.3953196947850739, 0.6330608335083314, 1.898310657149183, -0.029351127902527443, 0.7427328337621717, -0.33831406313040074, 0.0920082045447397, 0.0874176471196743, 0.05983564572452336, 0.11362012650575741], [-0.3481247266864322, 1.9105256958354058, -0.60342889957522, 0.2909082404547313, -0.252476791803004, -2.21433140332863, 1.014081570718048, 0.27340978305092917, -1.3309396416275998, -0.6865437031087761, -1.5716378217716929, 0.324499439758768, 1.59897831343026, 1.2550088075344976, -1.255871572366376, -1.6855427638682352], [0.5631978197948958, 1.7180851907698207, 0.08434885076115327, 1.0825546501385463, -1.361415899469512, -0.08002323669387834, 0.4712736382815932, -0.38139552372937824, -0.19773819667417775, 0.7119008201186471, -0.5968431318039679, -0.9665044207563379, 0.7722137763102623, 0.2683662165280098, -0.9716157988656038, 0.45727707819602825]], [[-0.5376127356953474, 0.6626300876609809, 0.014020724262976783, -0.2643422426079685, -0.750808791874266, 0.22371067624615593, -0.024490099505006135, 1.9131394543817049, -0.023186931500279076, -1.3266491793246087, 2.089943400389317, -2.017714388591036, 1.6711454174837166, -0.8429117726014337, 1.407823181786228, -0.16600172219306608], [0.5610540430931822, 0.04759712891396072, -0.09339558421005838, 0.5373561080684924, -0.6264275124884702, 0.49340285858683025, -1.0658353634238578, -0.7171913017637742, -0.49420982623205867, -0.08579065848748894, 1.2279667198701043, -0.8239346781917004, -0.8485469505140043, -1.1130437267387114, -0.3960882256008402, 0.5714571320641562], [-0.7374594503673815, -0.2848802346472962, -0.5806098743632441, 0.09148518728344766, -2.0073355821449748, 0.6005646180927717, 0.9309711472595998, -0.8293756561812254, -0.06355302824577985, -0.9029276378925517, 1.0076122042921374, 0.638022103029592, 2.905365754466288, -0.19979401144998388, -0.8672835241573296, -0.4707098575811255], [-1.2472258281739477, 0.21930308734331058, 0.32106353464181553, 0.9361309534630777, 0.41775047298731166, -0.3991506082722122, 1.308841835405336, 0.5965915157528774, 0.8488511521670594, -1.1523891832160071, -0.8151179060392977, -0.6208843954177016, -0.37051363122588304, -2.163180399362114, 1.2257990067736229, -0.18498699546738706], [0.8749902772222725, 0.802044669624543, 1.5252804711174976, 0.6059076613181522, 0.41080477286055705, -0.7030293439623367, 0.16296340347886626, 1.7967351326579435, -1.717788434249212, -0.6591092005876211, 0.32364641617881873, 1.6668239959895288, -1.0442172865679762, -0.39469683180605447, 0.2117067351592105, 3.165755800696517], [-0.80280893210664, -0.09042974238578648, 0.8122257890093711, -0.6343643651294132, 0.4578946078635953, 0.487082731292979, -0.28431569175490745, 0.4291280141007714, 1.8021716158207064, 0.20716255069019446, -0.343070423147098, -0.6623713896275052, 0.31705955904387295, 1.125269154091136, 0.23556134312775395, 0.4000979885722656], [2.7388393432137237, -0.6714810579558491, -0.9863119291726956, 0.36592425989631094, -0.12245718104423046, 0.6658942999413479, 1.331332445583354, 0.45100369130048223, -0.10280538971522843, 0.6117676927688177, 0.5766280062890846, 0.0021278608310701603, -0.48097402514264154, 0.7603778431664915, -0.108093126532051, -1.051913142309373]], [[0.7726398837103202, -0.4286010945500254, -0.4484858356809511, 0.7385463576161955, -0.6942186643905462, -0.08935070946846031, -0.8946296328413652, -1.4394504017367966, 0.3762050934336387, 1.6374032133317131, -0.7905504412515537, 1.2424164987637685, 1.1459267142565144, 1.5842377400820047, -0.9880662984708216, -0.5713481207529731], [1.724004292521382, 0.030054856813919305, 0.07493271159241456, 0.7916692976289752, 0.6963565304970232, -0.45971420467486895, -0.12285455181469986, 0.7264451323418675, 0.3592683694565223, -0.6347515129168594, 1.3690040206663212, 0.9434155480770531, -0.22143367827689114, -0.45325187658295996, 1.0176525463328299, -2.3092750447333943], [0.20587649264939326, -0.49152969477066893, -0.5723394726316076, 0.9834453183127081, 0.4088198305402953, 3.1559714470371008, -0.10617671130433942, 0.4767857945023709, -1.2829908919238033, 1.1209995390163658, -0.47676593542163204, 0.33706962021072734, 1.412172035526136, 1.2997242638605548, 0.6121462421466726, 1.399351671328819], [-0.3191929041625969, -0.8055963855196359, -1.3934029093372142, 0.678791910851508, 2.0840621539377424, -0.20035045317454983, 0.1331166119656035, -0.1918406459375135, 0.6065206993864899, -1.6988175345263283, 0.9740676968406007, 1.1983348117509025, -0.5592543679454854, -0.8869277896410422, -0.7598756353801258, 0.6695794546170647], [1.5612692495129983, -0.8750403302933807, -1.4243886453147128, 0.2175355482384409, 0.38094536017736325, 0.8203314322710019, -0.5792166912317633, 0.8422802138335908, 1.2073330236066813, -1.52617113438611, -0.19089399519207487, 0.7163811518815045, 0.8678210404240011, -0.4507667889153033, -1.841683702154883, -0.6826008245873167], [-1.3279404923210578, -0.4317360283869616, 1.1153472458055076, -0.5452500826413226, 0.16882159586740725, -1.1250587431946413, 1.0099464322267204, -0.5457932787098029, -0.9638190043615962, 1.9423645288166373, 0.41246453631946184, 0.7589519583252967, -1.916076233765082, 1.4979461716480538, 1.7902589188972187, -0.06238186965331408], [-0.22632982634519216, -0.5087815165390321, 1.1465881578236767, -1.4962526184200344, 0.39555401041219795, 2.06342647402826, 0.20564202087742098, 2.038755808695014, 0.09244606943396169, -0.23539557042329445, 0.9795228520308852, -0.005519108586433316, 0.1670966919961442, -0.33629786662270444, -1.138498373972039, -0.2406761316691224]]]], "dtype": "float64"}, {"__nd__": [7000, 7001], "dtype": "int64"}, {"__nd__": [6995, 6996, 6997, 6998, 6999, 7000, 7001], "dtype": "int64"}, 10], "e": {"__tup__": [{"__nd__": [[[[-1.339831516260504, 0.9298560939179474, -0.35477734036561465, -0.14361678454915994, 0.6902501993367263, 0.1790612724831544, -0.5218306454827637, 0.22897608182520854, -1.7202725262008771, -1.2645868693296982, 0.3247528649959841, -1.0487857298833259, 0.6854466141613622, 0.6521584987897013, 0.09434243617109991, 0.7128873475063768], [0.3717292041785651, -0.015923181606954717, 0.5723655863079334, -0.8805762482452653, 2.1344075013025283, 0.7559394501349244, 0.550611592126963, 0.0776396700443096, -1.1810401981224312, -0.9759997505607048, -0.3698057478971567, -1.0543470939889712, -0.34156052722203584, -0.8351430939597624, -0.4170599012110719, 0.3967821823193485]], [[-1.293882037723528, 0.42171761325733925, 0.7393659496695265, -1.5020630939441562, -0.3655426935305194, 0.39672114274317355, -2.3093623659069364, -0.5740420040642997, 0.3461514585661809, -1.7904588616059576, 0.43919349351729436, 1.024591077762065, -0.3232677308941149, 1.7133716327948385, 1.2759740433101456, 0.35307602330793086], [-0.21668943657756062, 0.17761155040087473, -2.501374747500403, -0.6152687844302596, 0.6101697587902938, 0.48058624097245145, 1.4436994734730726, -0.26280540868431584, -1.6501449129312493, -0.5240781801148511, 0.6243032398185351, -0.2989905966535996, 0.7467517137104908, 1.2923755482918649, 0.7663819416213754, -0.3931861881713645]], [[0.3645933635980844, 1.3979980075944627, -0.30529915672085, -0.8588772191988339, -0.12118767519375712, -0.03772916804419399, -0.9305819955601773, -0.12491515423617028, -0.47433890961314806, 0.5800454722314576, -1.1168312675166585, 0.0183330348372163, -0.34331274239099885, -1.983856891778598, -1.1801191577307932, -0.4033017548097564], [0.3788405460406362, 1.422363846596868, 0.07945035488573077, 0.35484181422879224, 0.8379130977852653, 1.466524871521102, -1.0894998916834455, 1.3421334037269734, 2.4644793907848763, -0.5721128891344751, 0.5544719540727608, 0.9869177298341354, 0.10338523921171394, 0.8337698290032067, 1.9270332857269237, 1.7130171495621231]], [[0.3926801701225531, -0.8500322671987027, 0.9592740533825005, -0.9254214715586108, 0.7834994020543339, 2.8640087034903443, -0.6812442868216984, 0.2457812226285533, 0.25244620752331626, 0.9054073837301613, -2.088412931573151, 0.24372955871343904, -1.2410871583938738, -0.9626480973950894, -1.8295130964672903, -1.6892691709142154], [0.7139481504768167, -0.31231662004942995, -0.11034549974083302, 1.0046823754328393, -0.03367407800285205, 0.5684566575558192, 1.2885239018848778, -0.2907113753457536, 1.9575152970733973, 1.2010959049570942, 0.912731508619598, 1.20625446188279, 1.1438534923955528, 0.498781081889901, -0.9924858329137045, 0.9059666097190682]], [[0.07347148979896923, -0.33865235065849475, -1.1419395769723677, -0.3624006640834077, -0.23792120747198367, 0.9021082545702153, 0.0875743530368468, 0.6586649835840813, 0.14933675998075813, -1.3971931250867098, 0.8423605426473773, 0.3437347798979491, 0.07454426859255363, 0.38411597350359655, -0.8126967496866394, -0.988282585949805], [0.11721355004034098, 1.42190717987853, 0.3786664532225795, 2.3362519651288114, 0.9898802742166954, 0.4789997568819985, -1.0330157131371775, -2.4931253512912526, 0.8200747541211891, -1.5459290909490473, 0.15800957363487542, 0.3047770205792349, 0.12683901406770667, 0.16268457466540617, 0.5384943889153253, -0.9986363697499262]], [[-0.6682267915799541, -1.2542755082677393, 0.16732078014704646, 0.3660195896379874, 0.7851211120702731, 0.8176569472606149, -0.3821717275525188, -0.1878218341166027, -0.6956644576357331, -0.34446862269756967, 1.4736863204876074, -0.12171205912006818, 0.3532571718128816, 1.045571670839754, -1.8825315671085434, 0.15230404541692436], [0.09219224631413436, -0.20399491016952206, -0.8476092569437745, -1.1601903427257008, 0.8369965185114774, 0.46615351659791865, 0.42415687485565606, -0.13761688156942542, -1.2063837568486926, -1.5597821312806537, 0.061488566602324234, -1.071458094300794, -1.238651024094895, -0.29998801234913414, -0.2925278571670005, 0.2874945668536115]]]], "dtype": "float64"}, {"__nd__": [[[[-0.3150693922666694, -0.09340829267286983, 0.2599106652158876, 1.918520552724066, 0.3158579327812856, -1.2145168110781581, -0.5518005338463382, -1.3611244663059732, 0.2195448240064919, -2.5152264944790708, 0.9134012445241502, -2.396483760058587, 1.1814470254959588, 0.7569620938110935, 0.653281059632514, -1.074377365542198], [-0.6128386973438165, 0.6116435777776623, 0.3910523177627448, -1.728924748612215, 0.8297905347147311, -0.9025529229276804, 0.15971867640493445, -1.078399894504382, 0.7186000471073326, -0.9623551430195312, 1.5076469684093838, -0.01707683380754581, 0.29611499026283156, -0.35429405231554517, -1.6708150408584765, 0.13019258819129437], [0.582600084204626, 0.3674818127243543, -0.01501157913181611, 1.4010513315265998, -0.708445707010743, -1.3591191255697184, -0.03513769562046609, 0.6533642244429578, -1.159632344645434, 0.10981316497014282, 0.25619616215093155, -0.7707925907281221, 2.9585937751357627, 0.252086333704382, -1.040342565011859, -1.4854231088889078], [-1.002604727013726, -0.5109421716661603, 0.3676680971080823, 0.40782003398385913, 0.6897097154627266, -1.64692406448387, -1.2163120802883163, -0.5371355479958422, -2.3308898500295996, 0.3278894940169413, -0.27574887506311774, 1.3840476434006113, -2.049071153160133, -0.7857514616136716, -0.35565023747576796, 1.1121408717949874], [1.5824382943027326, -0.4956718251607497, -1.6374719033818248, -0.3154583981102011, 0.28038747726492613, 0.6468345487798189, 0.26690941218633635, 0.8390178510130399, -1.9331781154416052, 1.018895752476893, 0.7427328337621717, -0.33831406313040074, 0.0920082045447397, 0.0874176471196743, 0.05983564572452336, 0.11362012650575741], [0.8223241342121641, -1.2913657371793579, -0.5740673597231376, 0.25949123856636597, -0.5831779445100025, -2.0852427694132065, -1.7351784920741031, 0.3306448439202142, 1.3374199905517956, 0.44157702806479543, -1.5716378217716929, 0.324499439758768, 1.59897831343026, 1.2550088075344976, -1.255871572366376, -1.6855427638682352], [0.10204942522170991, -1.1498323197130296, 0.05668600669266924, -0.9183711859345438, 1.0770281079756603, 0.5596261407802597, -1.3608090245971676, -0.3864763521753597, 0.6063161962258026, 1.0955667408443686, -0.5968431318039679, -0.9665044207563379, 0.7722137763102623, 0.2683662165280098, -0.9716157988656038, 0.45727707819602825]], [[-0.08722936669233672, -0.6142043979344597, 0.4362785721422238, 0.2502334416578367, -1.0468007047341878, -0.5757299348086754, 0.24986267349352556, 1.8627829074023436, -0.08830447154862983, 1.1081155948394372, 2.089943400389317, -2.017714388591036, 1.6711454174837166, -0.8429117726014337, 1.407823181786228, -0.16600172219306608], [-0.6897605417006275, 0.1555328196808708, -0.23238420489043995, -0.2872859062959903, 0.10204714391471235, -0.2871557334369609, 1.0554999055461052, -0.6848968387961747, 0.6711644710996928, 0.6239854532800099, 1.2279667198701043, -0.8239346781917004, -0.8485469505140043, -1.1130437267387114, -0.3960882256008402, 0.5714571320641562], [0.9510145594917836, 0.2545349880561166, -0.7143481230905616, -0.05773642641237657, -0.27467626082937036, -0.009777989349859073, -0.9397212166264134, -0.7174110149078644, 0.09526296214492078, 2.1838560873721335, 1.0076122042921374, 0.638022103029592, 2.905365754466288, -0.19979401144998388, -0.8672835241573296, -0.4707098575811255], [-0.5245095376212918, -0.05302104438428168, 0.40518854781304275, -1.198834691954074, -1.2241888749343053, 1.1999096713888782, -1.3260277384270347, 0.5429783335144871, -0.39960583320869547, -0.062272672669947404, -0.8151179060392977, -0.6208843954177016, -0.37051363122588304, -2.163180399362114, 1.2257990067736229, -0.18498699546738706], [0.46332463820120456, -0.7240492322856462, 1.7330127116803573, 0.13051311764057927, -0.7503393513617649, -1.0223446206350115, -0.3815565910492499, 1.5973116770543148, 1.8168344792545434, -0.2004404088557816, 0.32364641617881873, 1.6668239959895288, -1.0442172865679762, -0.39469683180605447, 0.2117067351592105, 3.165755800696517], [-0.93892722432029, -0.040180173161461205, 0.8500160558914115, -0.13053569435165646, 0.0647427007377471, 0.012939707269793821, 0.2956323807886177, 0.34833071863500087, -1.9060957775295526, -0.4983895835791571, -0.343070423147098, -0.6623713896275052, 0.31705955904387295, 1.125269154091136, 0.23556134312775395, 0.4000979885722656], [-0.5579689278322328, 1.3070145149743517, -0.9513021241508234, -0.2963268604713376, 0.6209403842992407, 2.762847596629881, -0.7176670185419008, 0.5208068929854485, 0.23803340920627783, -0.06073310671530957, 0.5766280062890846, 0.0021278608310701603, -0.48097402514264154, 0.7603778431664915, -0.108093126532051, -1.051913142309373]], [[-0.09992160710350319, 0.7089794292600343, -0.7553754316477145, -0.828026665217156, 1.7692178981755862, 0.7713440293515388, 0.6938366141199754, -1.3048237279915227, -0.03678092389973575, 0.18137493154340334, -0.7905504412515537, 1.2424164987637685, 1.1459267142565144, 1.5842377400820047, -0.9880662984708216, -0.5713481207529731], [-1.4851465756488595, -0.006194433676516321, 0.2160974658856536, -0.8693710330487323, -0.8113330424196914, 0.9888718822225985, 0.1263256280852116, 0.6975953890383624, -0.0028363702302762195, -0.4791248219348492, 1.3690040206663212, 0.9434155480770531, -0.22143367827689114, -0.45325187658295996, 1.0176525463328299, -2.3092750447333943], [1.806656724293646, 0.4947148500446612, -0.48200980578604746, -0.37674587128236736, 0.9518609911241973, -2.595868329731618, 0.09017844537382724, 0.5679469278881437, 1.572034627908148, -0.7195375416849082, -0.47676593542163204, 0.33706962021072734, 1.412172035526136, 1.2997242638605548, 0.6121462421466726, 1.399351671328819], [-0.23191856348774204, 0.8159398857896869, -1.406468612455293, -0.8656586414057956, -2.2342723945717333, 0.29704914419129774, -0.03078427392379869, 0.014850702381300124, -0.2815331838011116, -1.4957683456820534, 0.9740676968406007, 1.1983348117509025, -0.5592543679454854, -0.8869277896410422, -0.7598756353801258, 0.6695794546170647], [1.7634547360669943, 0.6772614052463409, -1.3110383073410363, -0.6815900907947645, -1.570893976189255, 0.027068839826663083, 0.8015638115616076, 1.0097017025841801, -1.0200047513463038, 0.08130076534915576, -0.19089399519207487, 0.7163811518815045, 0.8678210404240011, -0.4507667889153033, -1.841683702154883, -0.6826008245873167], [-0.5744266595629225, 0.8229961786148154, 1.0571551026694073, 0.8821312001898971, 1.8088756010787184, -1.6429294384020214, -0.7273686026769645, -0.6513930232284986, 0.6693947052677361, -0.7274955352841694, 0.41246453631946184, 0.7589519583252967, -1.916076233765082, 1.4979461716480538, 1.7902589188972187, -0.06238186965331408], [-2.070715305901002, 0.5358507375420698, 1.2903192137908603, 1.340230417584077, -0.3402478504402252, -0.14523128423814374, 0.11836916573976075, 1.9509653963285174, -0.6716402323156417, -0.31000879019616123, 0.9795228520308852, -0.005519108586433316, 0.1670966919961442, -0.33629786662270444, -1.138498373972039, -0.2406761316691224]]]], "dtype": "float64"}, {"__nd__": [[[[1.5391610883247135, 0.4939248861376264, 0.645920867088608, 0.5671036295339338, 0.31179694315755135, -1.2615939869795476, -0.07471714665075371], [0.08508646527172804, 0.7740105139713788, -0.7339967111114203, 0.5059384446059553, 0.6306661502220725, -1.8730920550364503, 0.5982829685273202]], [[0.7578706571543364, 0.8383806734134861, -1.7350538447650092, 0.5277127137999809, -1.5536674643250807, -0.11729177900425682, 0.0312435532962334], [0.6043864276107886, -0.09749245922029287, 0.593739496329472, -0.7991358983816992, 1.9774058024574277, -1.020379739232305, -0.5998116157605694]], [[-0.9344055532941841, 0.19471907617964465, 0.6720433642723762, 1.502220391360822, -0.7233675628968925, -0.6668421561634058, 0.2520910850598956], [-0.015586006332080187, -0.5501509971776297, -0.4271367583769272, 0.10737978577761852, 2.8929441398604006, -0.8297559015557061, 1.7592791604698672]], [[-2.23498017716338, -0.5342511303754762, -0.3082996905919774, 1.7143087496581708, -1.1440023007042937, -0.5968289375892172, 1.9222143360113584], [0.09911589547361115, 0.5409298890756534, 1.9446173809754537, -1.9967814331811449, 1.4683317194148375, -1.2468064413807856, -0.15513244036903442]], [[0.48882079872512274, 1.2319927803915982, -0.463914603209369, 1.3216992547703803, 1.1650415037578588, -0.8722548870055998, 0.2887521134966633], [0.9544446725061605, -0.08418840636572847, -0.051526658211588965, -0.5472603882885418, -1.7134495462313748, 2.4850435036271676, -0.39460326513281496]], [[0.8072156540326275, -0.009883672998310208, -1.0583685480644758, -0.12866363100784237, -0.17312554215618345, -0.46536963931629916, 1.1131769694643956], [0.09328204095037246, -0.07449203890178083, -0.6556163741236692, 0.771034847694583, 0.13811403501728978, -0.17233415642650562, -0.5004708493556127]]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(rope_qk, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")